# FNO Wrapper Validation (M3)

This notebook profiles and validates the `ArbitrageFreeFNO` wrapper and its underlying projection layer, checking:
1. Execution speed (latency) on both CPU and GPU.
2. VRAM footprint and memory efficiency on the GPU.
3. Shape and type consistency.
4. Arbitrage elimination.

In [1]:
import os
import time
import torch
import numpy as np
from deepvol.arbitrage.projection_layer import DifferentiableArbitrageFreeProjection, ArbitrageFreeFNO
from deepvol.surrogates.fno_model import MirrorPaddedFNO2d
from deepvol.surrogates.normalizers import IVSurfaceNormalizer

# Set seeds
torch.manual_seed(42)
np.random.seed(42)

# Grid definition
T_grid = torch.tensor([0.1, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.0], dtype=torch.float64)
K_grid = torch.linspace(-0.5, 0.5, 11, dtype=torch.float64)

# Load/create normalizer
path = "/home/execorn/programming/derivatives-w3/artifacts/models/iv_normalizer_v2.npz"
if os.path.exists(path):
    print("Loading normalizer from artifacts...")
    norm = IVSurfaceNormalizer.load(path)
else:
    print("Normalizer artifact not found, creating a mock normalizer...")
    norm = IVSurfaceNormalizer()
    norm.mean = np.full((8, 11), 0.3)
    norm.std = np.full((8, 11), 0.1)

# Base FNO and projection
base_fno = MirrorPaddedFNO2d()
proj = DifferentiableArbitrageFreeProjection(T_grid=T_grid, K_grid=K_grid)

# Wrapper
wrapper = ArbitrageFreeFNO(base_fno=base_fno, projection_layer=proj, normalizer=norm)
print("ArbitrageFreeFNO wrapper successfully initialized!")

Loading normalizer from artifacts...
ArbitrageFreeFNO wrapper successfully initialized!


In [2]:
# Generate inputs
B = 128  # Batch size for profiling
spatial = torch.randn(B, 8, 11, 2, dtype=torch.float32)
theta = torch.randn(B, 6, dtype=torch.float32)

# CPU Profiling
print("Profiling forward pass on CPU...")
# Warmup
with torch.no_grad():
    for _ in range(10):
        _ = wrapper(spatial, theta)
    
    start_time = time.perf_counter()
    runs = 100
    for _ in range(runs):
        _ = wrapper(spatial, theta)
    end_time = time.perf_counter()
cpu_time_ms = (end_time - start_time) * 1000 / runs
print(f"Average CPU forward pass latency (Batch size={B}): {cpu_time_ms:.4f} ms")

# GPU Profiling
if torch.cuda.is_available():
    print("\nProfiling forward pass on GPU...")
    device = torch.device("cuda")
    wrapper_gpu = wrapper.to(device)
    spatial_gpu = spatial.to(device)
    theta_gpu = theta.to(device)
    
    with torch.no_grad():
        # Warmup
        for _ in range(10):
            _ = wrapper_gpu(spatial_gpu, theta_gpu)
        torch.cuda.synchronize()
        
        # Time with synchronization
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record()
        for _ in range(runs):
            _ = wrapper_gpu(spatial_gpu, theta_gpu)
        end_event.record()
        
        torch.cuda.synchronize()
        gpu_time_ms = start_event.elapsed_time(end_event) / runs
        print(f"Average GPU forward pass latency (Batch size={B}): {gpu_time_ms:.4f} ms")
else:
    print("\nGPU is not available for profiling.")

Profiling forward pass on CPU...


Average CPU forward pass latency (Batch size=128): 73.0254 ms

Profiling forward pass on GPU...


Average GPU forward pass latency (Batch size=128): 16.0372 ms


In [3]:
if torch.cuda.is_available():
    print("Profiling VRAM memory usage on GPU...")
    device = torch.device("cuda")
    
    # Clear cache and reset stats
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    
    # Initial memory allocated
    mem_init = torch.cuda.memory_allocated(device) / (1024 ** 2)
    print(f"Initial allocated memory: {mem_init:.4f} MB")
    
    # Load model and inputs to GPU
    wrapper_gpu = wrapper.to(device)
    spatial_gpu = spatial.to(device)
    theta_gpu = theta.to(device)
    
    mem_loaded = torch.cuda.memory_allocated(device) / (1024 ** 2)
    print(f"Allocated memory after loading model & inputs: {mem_loaded:.4f} MB")
    
    # Run forward pass under no_grad
    with torch.no_grad():
        out = wrapper_gpu(spatial_gpu, theta_gpu)
    
    mem_after_forward = torch.cuda.memory_allocated(device) / (1024 ** 2)
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    
    print(f"Allocated memory after forward pass: {mem_after_forward:.4f} MB")
    print(f"Peak VRAM memory allocated: {peak_mem:.4f} MB")
    print(f"Memory overhead of forward pass: {peak_mem - mem_loaded:.4f} MB")
    
    # Check that peak memory is well below 4 GB (as constrained by AGENTS.md)
    print(f"Peak memory is {peak_mem:.2f} MB, which is well within the 4 GB limit.")
else:
    print("GPU is not available for VRAM profiling.")

Profiling VRAM memory usage on GPU...
Initial allocated memory: 13.1992 MB
Allocated memory after loading model & inputs: 13.1992 MB
Allocated memory after forward pass: 13.2422 MB
Peak VRAM memory allocated: 42.8164 MB
Memory overhead of forward pass: 29.6172 MB
Peak memory is 42.82 MB, which is well within the 4 GB limit.


In [4]:
from deepvol.surrogates.fno_model import arbitrage_free_regularization

# Check raw model output (which has arbitrage violations)
# Let's generate an arbitrage-violating surface and project it
class ArbitraryFNO(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dummy = torch.nn.Parameter(torch.randn(1))
    def forward(self, spatial, theta):
        B = spatial.size(0)
        surf = torch.zeros(B, 8, 11, dtype=spatial.dtype, device=spatial.device)
        # Put high vol at short maturity and low/negative vol at long maturity to guarantee violations
        surf[:, 0, :] = 2.0
        surf[:, 7, :] = -2.0
        return surf

test_wrapper = ArbitrageFreeFNO(base_fno=ArbitraryFNO(), projection_layer=proj, normalizer=norm)

# Raw output
with torch.no_grad():
    raw_out = test_wrapper.base_fno(spatial, theta)
    raw_iv = norm.inverse_transform_tensor(raw_out)
    raw_penalty = arbitrage_free_regularization(raw_iv, T_grid, K_grid)
    print(f"Raw surface arbitrage penalty: {raw_penalty.mean().item():.6f}")

    # Projected output
    clean_out = test_wrapper(spatial, theta)
    clean_iv = norm.inverse_transform_tensor(clean_out)
    clean_penalty = arbitrage_free_regularization(clean_iv, T_grid, K_grid)
    print(f"Projected surface arbitrage penalty: {clean_penalty.mean().item():.6f}")

assert clean_penalty.mean().item() < 1e-6
print("Arbitrage-free guarantee verified successfully!")

Raw surface arbitrage penalty: 0.202792
Projected surface arbitrage penalty: 0.000000
Arbitrage-free guarantee verified successfully!
